In [20]:
import pandas as pd
import numpy as np
import textwrap
import matplotlib.pyplot as plt
import math
import seaborn as sns
import datamol as dm
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from multiprocessing import Pool
import os
from pathlib import Path
import hashlib
from sklearn.model_selection import GroupShuffleSplit

from itertools import islice
df_well=pd.read_pickle('/home/ethan2/GrowthCurve/data/df_well_fingerprints.pkl')

In [2]:
import pandas as pd

# Keep only the two columns, drop NaNs, strip whitespace, drop exact duplicate pairs
pairs = (
    df_well[["Compound", "Smiles"]]
    .dropna()
    .assign(
        Compound=lambda d: d["Compound"].astype(str).str.strip(),
        Smiles=lambda d: d["Smiles"].astype(str).str.strip(),
    )
    .drop_duplicates()
)

# 1) Does every Compound map to exactly one SMILES?
smiles_per_compound = pairs.groupby("Compound")["Smiles"].nunique()
compounds_with_multiple_smiles = (
    pairs.groupby("Compound")["Smiles"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="smiles")
)
compounds_with_multiple_smiles["n_smiles"] = compounds_with_multiple_smiles["smiles"].str.len()
compounds_with_multiple_smiles = compounds_with_multiple_smiles.query("n_smiles > 1")

compound_to_smiles_is_unique = compounds_with_multiple_smiles.empty

# 2) Do different Compounds share the same SMILES?
compounds_per_smiles = pairs.groupby("Smiles")["Compound"].nunique()
smiles_used_by_multiple_compounds = (
    pairs.groupby("Smiles")["Compound"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="compounds")
)
smiles_used_by_multiple_compounds["n_compounds"] = smiles_used_by_multiple_compounds["compounds"].str.len()
smiles_used_by_multiple_compounds = smiles_used_by_multiple_compounds.query("n_compounds > 1")

smiles_to_compound_is_unique = smiles_used_by_multiple_compounds.empty

# 3) Summary
print(f"Unique mapping Compound→SMILES: {compound_to_smiles_is_unique}")
print(f"Unique mapping SMILES→Compound: {smiles_to_compound_is_unique}")

if not compound_to_smiles_is_unique:
    print("\nCompounds with >1 distinct SMILES:")
    print(compounds_with_multiple_smiles.sort_values("n_smiles", ascending=False).head(20))

if not smiles_to_compound_is_unique:
    print("\nSMILES used by >1 distinct Compound:")
    print(smiles_used_by_multiple_compounds.sort_values("n_compounds", ascending=False).head(20))


Unique mapping Compound→SMILES: False
Unique mapping SMILES→Compound: True

Compounds with >1 distinct SMILES:
          Compound                                             smiles  \
198  Ciprofloxacin  [C1CNCCN1c(c2)c(F)cc3c2N(C4CC4)C=C(C3=O)C(=O)O...   

     n_smiles  
198         2  


This is fine, won't be looking at positive controls anyway 

In [3]:
df_no_nan = df_well.dropna(subset=["maccs_fp", "ecfp_fp", "rdkit_fp"]).copy()

def fp_key(r):
    fp = np.concatenate([r["maccs_fp"], r["ecfp_fp"], r["rdkit_fp"]])
    # Hash the raw bytes; consistent within the dataset as long as dtype/shape are consistent
    return hashlib.sha1(fp.tobytes()).hexdigest()

df_no_nan["fp_key"] = df_no_nan.apply(fp_key, axis=1)

collisions = (
    df_no_nan.groupby("fp_key")["Compound"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="compounds")
)
collisions["n_compounds"] = collisions["compounds"].str.len()
collisions = collisions.query("n_compounds > 1")

if collisions.empty:
    print("✅ No duplicate concatenated fingerprints across different compounds.")
else:
    print("⚠️ Fingerprint collisions found:")
    print(collisions[["fp_key", "n_compounds", "compounds"]])

⚠️ Fingerprint collisions found:
                                         fp_key  n_compounds  \
518    03ca64cfa8a528b5c12c4df9aa37beaaa0a1fe35            2   
758    05af9b879fa2548bcda815ace30094424b1b461f            2   
1364   09fd74b6e00786a948b799dcc61f7d1994d624ac            2   
1550   0b67db78e2d4a4e8b6fbcf760df6de62ab93ca98            2   
3464   199114a944f7cf3aa94c80623f4c8f9d9ddd0f4a            2   
...                                         ...          ...   
31606  e9d90a972beb85cfa57c43abf92122e53aae3520            2   
31896  ebee7015fc1038e342c9fc48b697890dda10dffb            2   
32337  ef5bf11086adc148b237ce12d8eb459c57ca3f4f            2   
33234  f5cee6bfb8f9e31436e8e4d160ff71e338910b6e            2   
34476  ff0aa0443ab7306378ce688f672f4c78691442eb            2   

                              compounds  
518               [Farnesol, Solanesol]  
758         [Florfenicol, UM0133378:01]  
1364   [Sulfadimethoxine, UM0118812:01]  
1550       [UM0119423:01, UM01

In [4]:
df_well[df_well['Compound'].isin(['Farnesol', 'Solanesol'])]['Smiles'].unique()

array(['C/C(C)=C\\CC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CC/C(C)=C/CO',
       'C/C(C)=C\\CC/C(C)=C/CC/C(C)=C/CO'], dtype=object)

They have different SMILES, but identical fingerprints

['Florfenicol', 'UM0133378:01']

In [5]:
maccs_fp_Florfenicol=df_well[df_well['Compound']=="Florfenicol"]['maccs_fp'].values[0]
maccs_fp_UM0133378_01=df_well[df_well['Compound']=="UM0133378:01"]['maccs_fp'].values[0]

ecfp_fp_Florfenicol=df_well[df_well['Compound']=="Florfenicol"]['ecfp_fp'].values[0]
ecfp_fp_UM0133378_01=df_well[df_well['Compound']=="UM0133378:01"]['ecfp_fp'].values[0]


rdkit_fp_Florfenicol=df_well[df_well['Compound']=="Florfenicol"]['rdkit_fp'].values[0]
rdkit_fp_UM0133378_01=df_well[df_well['Compound']=="UM0133378:01"]['rdkit_fp'].values[0]

print(np.array_equal(maccs_fp_Florfenicol, maccs_fp_UM0133378_01))
print(np.array_equal(ecfp_fp_Florfenicol, ecfp_fp_UM0133378_01))
print(np.array_equal(rdkit_fp_Florfenicol, rdkit_fp_UM0133378_01))

True
True
True


# Plot drawings

In [32]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

def _pick_rep_smiles(series: pd.Series) -> str:
    vc = series.value_counts()
    return vc.index[0]

def draw_collision_groups(collisions: pd.DataFrame,
                          df_no_nan: pd.DataFrame,
                          #num_groups: int = 2,
                          outdir: str = "/home/ethan2/GrowthCurve/plots/collision_groups/drawings/",
                          molsPerRow: int = 3,
                          subImgSize=(300, 300)) -> None:
    os.makedirs(outdir, exist_ok=True)

    # Try to import PIL to detect/save PNGs if available
    try:
        from PIL import Image as PILImage
    except Exception:
        PILImage = None

    for idx, row in collisions.iterrows():
        fp_key = row["fp_key"]

        g = (
            df_no_nan.loc[df_no_nan["fp_key"] == fp_key, ["Compound", "Smiles"]]
            .dropna()
            .drop_duplicates()
        )
        if g.empty:
            print(f"[{idx}] No rows found for fp_key={fp_key}")
            continue

        # One representative SMILES per compound
        rep = g.groupby("Compound", as_index=True)["Smiles"].apply(_pick_rep_smiles)

        mols, legends = [], []
        for comp, smi in rep.items():
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                print(f"[{idx}] Skipping invalid SMILES for {comp}: {smi}")
                continue
            rdMolDraw2D.PrepareMolForDrawing(mol)
            mols.append(mol)
            legends.append(f"{comp}\n{smi}")

        if not mols:
            print(f"[{idx}] No valid molecules to draw for fp_key={fp_key}")
            continue

        # Try PNG via PIL first
        img = Draw.MolsToGridImage(
            mols, molsPerRow=molsPerRow, subImgSize=subImgSize, legends=legends
        )

        base = os.path.join(outdir, f"collision_{idx:03d}_n{len(mols)}")
        if PILImage is not None and hasattr(img, "save"):
            out_png = base + ".png"
            img.save(out_png)
            print(f"Wrote {out_png}")
        else:
            # Fall back to SVG (handle both string and IPython SVG object)
            svg_obj = Draw.MolsToGridImage(
                mols, molsPerRow=molsPerRow, subImgSize=subImgSize,
                legends=legends, useSVG=True
            )
            svg_str = (
                svg_obj
                if isinstance(svg_obj, str)
                else getattr(svg_obj, "data", str(svg_obj))
            )
            out_svg = base + ".svg"
            with open(out_svg, "w", encoding="utf-8") as f:
                f.write(svg_str)
            print(f"Wrote {out_svg} (saved as SVG)")

draw_collision_groups(collisions, df_no_nan)


Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_518_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_758_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_1364_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_1550_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_3464_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_3699_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_3947_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_5171_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_5920_n2.svg (saved as SVG)
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/drawings/collision_7210_n2.svg (saved as SVG)
Wr

# Plot OD values

In [23]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import islice

def _mean_abs_pairwise(vals: np.ndarray) -> float:
    """Mean(|xi-xj|) over all unordered pairs. Returns NaN if <2 values."""
    x = np.asarray(vals, dtype=float)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 2:
        return np.nan
    diffs = np.abs(x[:, None] - x[None, :])
    upper = diffs[np.triu_indices(n, 1)]
    return float(upper.mean())

def save_od_diff_heatmaps_for_collisions(
    collisions: pd.DataFrame,
    df_well: pd.DataFrame,
    outdir: str = "/home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/",
    num_groups: int | None = None,
    agg: str = "mean",          # aggregate replicates per (Compound,T,C)
    fontsize_cap: int = 10,     # max font size for annotations (pt)
    min_fontsize: int = 5,      # minimum font size (pt)
    max_name_len: int = 16,     # upper bound for name length before truncation
    vmin: float = 0.0,          # fixed legend min (requested)
    vmax: float = 0.2,           # fixed legend max (requested)
    title_width: int = 50        # max width for title text wrapping
) -> None:
    """
    For each collision row, make a (Timepoint x Concentration) heatmap where
    color = mean absolute pairwise ΔOD across compounds. Cell text shows each
    compound's aggregated OD. Colorbar is fixed to [vmin, vmax].
    """
    need_cols = {"Compound", "Timepoint", "Concentration", "OD"}
    miss = need_cols - set(df_well.columns)
    if miss:
        raise ValueError(f"df_well missing required columns: {miss}")

    os.makedirs(outdir, exist_ok=True)
    rows_iter = collisions.itertuples(index=False)
    if num_groups is not None:
        rows_iter = islice(rows_iter, num_groups)

    for idx, row in enumerate(rows_iter):
        fp_key = getattr(row, "fp_key", f"group_{idx}")
        compounds = list(row.compounds) if hasattr(row, "compounds") else []
        if not compounds:
            print(f"[{idx}] Skipping: no compounds listed.")
            continue

        sub = df_well[df_well["Compound"].isin(compounds)].copy()
        if sub.empty:
            print(f"[{idx}] No matching rows in df_well for fp_key={fp_key}.")
            continue

        aggfunc = "median" if agg.lower() == "median" else "mean"
        per_comp = (
            sub.groupby(["Compound", "Timepoint", "Concentration"], as_index=False)["OD"]
               .agg(aggfunc)
        )

        # Map (T,C) -> {compound -> OD}
        cell_map: dict[tuple, dict] = {}
        for r in per_comp.itertuples(index=False):
            key = (getattr(r, "Timepoint"), getattr(r, "Concentration"))
            cell_map.setdefault(key, {})[getattr(r, "Compound")] = float(getattr(r, "OD"))

        t_vals = sorted(per_comp["Timepoint"].unique())
        c_vals = sorted(per_comp["Concentration"].unique())

        # Matrix of mean absolute pairwise differences
        diff_mat = pd.DataFrame(index=t_vals, columns=c_vals, dtype=float)
        for t in t_vals:
            for c in c_vals:
                od_map = cell_map.get((t, c), {})
                if not od_map:
                    diff_mat.loc[t, c] = np.nan
                else:
                    vals = np.array(list(od_map.values()), dtype=float)
                    diff_mat.loc[t, c] = _mean_abs_pairwise(vals)

        data = diff_mat.values.astype(float)
        finite = np.isfinite(data)
        if not finite.any():
            print(f"[{idx}] All cells are NaN for fp_key={fp_key}; skipping.")
            continue
        masked = np.ma.array(data, mask=~finite)

        # Figure size scales with grid size
        nT, nC = diff_mat.shape
        fig_w = max(6, 0.9 * nC + 3)
        fig_h = max(6, 0.9 * nT + 3)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(masked, aspect="auto", origin="lower", vmin=vmin, vmax=vmax)
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Mean absolute pairwise ΔOD")

        # Axes labels/ticks
        ax.set_xticks(range(nC))
        ax.set_yticks(range(nT))
        ax.set_xticklabels([f"{c:g}" for c in diff_mat.columns], rotation=45, ha="right")
        ax.set_yticklabels([f"{t:g}" for t in diff_mat.index])
        ax.set_xlabel("Concentration")
        ax.set_ylabel("Timepoint")

        comp_order = sorted(set(per_comp["Compound"]), key=str)
        short_fp = str(fp_key)[:10]
        
        comp_line = ", ".join(map(str, comp_order))
        comp_wrapped = textwrap.fill(comp_line, width=title_width)
        ax.set_title(
            "ΔOD (mean |pairwise differences|) by (Timepoint, Concentration)\n"
            f"Compounds: {comp_wrapped}"
        )

        # --- Visibility helpers: auto font + auto truncation per cell ---
        def short(name: str, max_len: int) -> str:
            name = str(name)
            return name if len(name) <= max_len else name[:max_len-1] + "…"

        # Cell size in points (1 pt = 1/72 in)
        cell_h_pt = fig_h * 72.0 / max(nT, 1)
        cell_w_pt = fig_w * 72.0 / max(nC, 1)

        # Approximate per-line font size to fit all lines vertically
        lines_per_cell = max(1, len(comp_order))
        fs_auto = 0.9 * cell_h_pt / (lines_per_cell + 0.5)
        fs = max(min_fontsize, min(fontsize_cap, fs_auto))

        # Approximate max characters that fit horizontally (heuristic)
        # Character width ~ 0.6 * fontsize (pts)
        approx_char_w = 0.6 * fs
        max_chars_horiz = max(3, int(0.9 * cell_w_pt / approx_char_w) - 8)  # minus space for ": 0.000"
        name_len = max(3, min(max_name_len, max_chars_horiz))

        # Annotations
        for yi, t in enumerate(diff_mat.index):
            for xi, c in enumerate(diff_mat.columns):
                od_map = cell_map.get((t, c), {})
                if not od_map:
                    continue
                lines = []
                for comp in comp_order:
                    v = od_map.get(comp, None)
                    if v is None or (isinstance(v, float) and (math.isnan(v))):
                        lines.append(f"{short(comp, name_len)}: –")
                    else:
                        lines.append(f"{short(comp, name_len)}: {v:.3f}")
                ax.text(xi, yi, "\n".join(lines), ha="center", va="center", fontsize=fs)

        fig.tight_layout()
        out_path = os.path.join(outdir, f"ODHM_diff_{idx:03d}_{short_fp}.png")
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Wrote {out_path}")


# Regenerate all figures with fixed legend [0, 0.2] and auto-fitting text
save_od_diff_heatmaps_for_collisions(
    collisions=collisions,
    df_well=df_well[df_well['Timepoint'] != 0],  # limit to first 24h
    outdir="/home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/",
    num_groups=10,         # or an int like 10
    agg="mean",
    fontsize_cap=7,         # you can lower/raise this if needed
    min_fontsize=7,
    max_name_len=10,         # upper bound (auto-truncation will reduce if needed)
    vmin=0.0,
    vmax=0.1,
    title_width=50          # max width for title text wrapping (auto-wraps long titles
)


Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_000_03ca64cfa8.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_001_05af9b879f.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_002_09fd74b6e0.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_003_0b67db78e2.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_004_199114a944.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_005_1b5185ece2.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_006_1d398305e7.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_007_26904160c2.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_008_2b9e62fa9f.png
Wrote /home/ethan2/GrowthCurve/plots/collision_groups/OD_hm/ODHM_diff_009_353a17a9b1.png


In [14]:
df_well["Timepoint"].unique()

array([ 0.  ,  2.08,  4.16,  6.24,  8.32, 10.4 , 12.48])

# Check which are actually active!!!